In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from nystrom_attention import NystromAttention
from torch.autograd import Function


class TransLayer(nn.Module):

    def __init__(self, norm_layer=nn.LayerNorm, dim=512):
        super().__init__()
        self.norm = norm_layer(dim)
        self.attn = NystromAttention(
            dim = dim,
            dim_head = dim//8,
            heads = 8,
            num_landmarks = dim//2,    # number of landmarks
            pinv_iterations = 6,    # number of moore-penrose iterations for approximating pinverse. 6 was recommended by the paper
            residual = True,         # whether to do an extra residual with the value or not. supposedly faster convergence if turned on
            dropout=0.1
        )

    def forward(self, x):
        x = x + self.attn(self.norm(x))

        return x


class PPEG(nn.Module):
    def __init__(self, dim=512):
        super(PPEG, self).__init__()
        self.proj = nn.Conv2d(dim, dim, 7, 1, 7//2, groups=dim)
        self.proj1 = nn.Conv2d(dim, dim, 5, 1, 5//2, groups=dim)
        self.proj2 = nn.Conv2d(dim, dim, 3, 1, 3//2, groups=dim)

    def forward(self, x, H, W):
        B, _, C = x.shape
        cls_token, feat_token = x[:, 0], x[:, 1:]
        cnn_feat = feat_token.transpose(1, 2).view(B, C, H, W)
        x = self.proj(cnn_feat)+cnn_feat+self.proj1(cnn_feat)+self.proj2(cnn_feat)
        x = x.flatten(2).transpose(1, 2)
        x = torch.cat((cls_token.unsqueeze(1), x), dim=1)
        return x


class TransMIL(nn.Module):
    def __init__(self, n_classes, input_dims):
        super(TransMIL, self).__init__()
        self.pos_layer = PPEG(dim=512)
        self._fc1 = nn.Sequential(nn.Linear(input_dims, 512), nn.ReLU())
        self.cls_token = nn.Parameter(torch.randn(1, 1, 512))
        self.n_classes = n_classes
        self.layer1 = TransLayer(dim=512)
        self.layer2 = TransLayer(dim=512)
        self.norm = nn.LayerNorm(512)
        self._fc2 = nn.Linear(512, self.n_classes)
        self.loss = nn.CrossEntropyLoss()


    def forward(self, data, label):

        h = data.float().unsqueeze(0) #[B, n, 1024]        
        h = self._fc1(h) #[B, n, 512]
        
        #---->pad
        H = h.shape[1]
        _H, _W = int(np.ceil(np.sqrt(H))), int(np.ceil(np.sqrt(H)))
        add_length = _H * _W - H
        h = torch.cat([h, h[:,:add_length,:]],dim = 1) #[B, N, 512]

        #---->cls_token
        B = h.shape[0]
        cls_tokens = self.cls_token.expand(B, -1, -1).cuda()
        h = torch.cat((cls_tokens, h), dim=1)

        #---->Translayer x1
        h = self.layer1(h) #[B, N, 512]

        #---->PPEG
        h = self.pos_layer(h, _H, _W) #[B, N, 512]
        
        #---->Translayer x2
        h = self.layer2(h) #[B, N, 512]

        #---->cls_token
        h = self.norm(h)[:,0]

        #---->predict
        logits = self._fc2(h) #[B, n_classes]
        Y_hat = torch.argmax(logits, dim=1)
        Y_prob = F.softmax(logits, dim = 1)
        loss = self.loss(logits, label)
        results_dict = {'logits': logits, 'Y_prob': Y_prob, 'Y_hat': Y_hat, 'Loss': loss}

        return results_dict

/home/andrewtal/.conda/envs/transmil/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class ReverseLayerF(Function):

    @staticmethod
    def forward(ctx, x, alpha=1.0):
        ctx.alpha = alpha

        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        output = grad_output.neg() * ctx.alpha
        return output, None
    
    
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, key_concepts, confuse_concepts, concept_dim):
        super(Encoder, self).__init__()
        self.concept_dim = concept_dim
        self.key_concepts = key_concepts
        self.confuse_concepts = confuse_concepts

        self.eps_mu_key = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, key_concepts*concept_dim),
        )

        self.eps_mu_confuse = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, confuse_concepts*concept_dim),
        )

    def forward(self, x):
        N = x.shape[0]
        eps_mu_key = self.eps_mu_key(x).reshape(N, self.key_concepts, self.concept_dim)
        eps_mu_confuse = self.eps_mu_confuse(x).reshape(N, self.confuse_concepts, self.concept_dim)
        
        return eps_mu_key, eps_mu_confuse
    

class Decoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(Decoder, self).__init__()
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )
    
    def forward(self, x):
        x = self.decoder(x)
        
        return x
    
    
class CausalBlock(nn.Module):
    def __init__(self, concepts):
        super(CausalBlock, self).__init__()
        self.I = nn.Parameter(torch.eye(concepts)); self.I.requires_grad=False
        # self.A = nn.Parameter((torch.zeros(concepts, concepts) + 0.25).fill_diagonal_(0.))
        self.A = nn.Parameter(torch.triu(torch.zeros(concepts, concepts)+0.1, diagonal=1))
        
    def forward(self, eps):
        z = torch.matmul(torch.inverse(self.I - self.A.T), eps)
        return z
    
    
class CDG(nn.Module):
    def __init__(self, n_classes=2, input_dim=768, hidden_dim=256, key_concepts=4, confuse_concepts=12, concept_dim=16, cls_alpha=1., ha_alpha=1.):
        super(CDG, self).__init__()

        concept_dim = concept_dim
        concepts = key_concepts + confuse_concepts
        latent_dim = (key_concepts+confuse_concepts)*concept_dim
        
        self.ha_alpha = ha_alpha
        self.cls_alpha = cls_alpha
        self.confuse_concepts = confuse_concepts
        self.encoder = Encoder(input_dim, hidden_dim, key_concepts, confuse_concepts, concept_dim)
        self.decoder = Decoder(input_dim, hidden_dim, latent_dim)
        self.causal = CausalBlock(concepts)
        
        self.classifier = TransMIL(n_classes=n_classes, input_dims=key_concepts*concept_dim)
        self.grl_classifier = TransMIL(n_classes=n_classes, input_dims=confuse_concepts*concept_dim)
    
        # loss
        self.mse_loss = nn.MSELoss()
    
    def matrix_poly(self, matrix, d):
        x = torch.eye(d).to(matrix.device) + torch.div(matrix, d)
        return torch.matrix_power(x, d)
        
    def h_A(self, A):
        m = A.size()[0]
        expm_A = self.matrix_poly(A*A, m)
        h_A = torch.trace(expm_A) - m
        return h_A
        
    def kl_normal(self, mu, logvar):
        kl = -0.5 * (1. + logvar - mu.pow(2) - logvar.exp()).sum(dim=-1).mean()
        return kl

    def sparse_a(self, A):
        loss = torch.mean(torch.abs(A))
        return loss

    def forward(self, data, label):
        # causal block
        data = data.squeeze(0)
        eps_mu_key, eps_mu_confuse = self.encoder(data)
        eps_mu = torch.cat([eps_mu_confuse, eps_mu_key], dim=1)
        z = self.causal(eps_mu)
        z_logvar = torch.zeros_like(z).to(z.device)
        
        # recon x
        recon_x = self.decoder(z.flatten(start_dim=1))

        # classification brunch
        z_key = z[:, self.confuse_concepts:].flatten(start_dim=1)
        res_key = self.classifier(z_key, label)
        loss_cls = res_key['Loss']

        # grl brunch
        z_confuse = z[:, :self.confuse_concepts].flatten(start_dim=1)
        z_confuse = ReverseLayerF.apply(z_confuse)
        res_grl = self.grl_classifier(z_confuse, label)
        loss_grl = res_grl['Loss']
        
        # ha
        loss_a = self.h_A(self.causal.A) # + self.sparse_a(self.causal.A)

        # loss
        elbo = self.kl_normal(z.flatten(start_dim=1), z_logvar.flatten(start_dim=1)) + self.mse_loss(recon_x, data)
        
        loss = loss_cls + loss_grl + elbo + loss_a
        
        return {'logits': res_key['logits'], 'Y_prob': res_key['Y_prob'], 'Y_hat': res_key['Y_hat'], 'Loss': loss}

In [3]:
x = torch.randn(1, 65, 768).cuda()
y = torch.LongTensor([1]).cuda()

In [4]:
model =CDG(n_classes=2).cuda()

In [5]:
model(data=x, label=y)

{'logits': tensor([[ 0.2347, -0.7934]], device='cuda:0', grad_fn=<AddmmBackward>),
 'Y_prob': tensor([[0.7366, 0.2634]], device='cuda:0', grad_fn=<SoftmaxBackward>),
 'Y_hat': tensor([0], device='cuda:0'),
 'Loss': tensor(12.7376, device='cuda:0', grad_fn=<AddBackward0>)}